In [1]:
import cv2
from ultralytics import YOLO

model = YOLO(r"runs/detect/koi-detect/exp_real_100/weights/best.pt")
video_path = r"../data/real_test_data/video.mp4"

cap = cv2.VideoCapture(video_path)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
print(f'ขนาด: {w}x{h}, FPS: {fps}')

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(
        frame,
        conf=0.3,
        verbose=False,
        device=0,
        imgsz=max(w, h)  # ใช้ขนาดจริงของวิดีโอเลย
    )

    for i, box in enumerate(results[0].boxes.xyxy):
        x1, y1, x2, y2 = map(int, box)
        conf = float(results[0].boxes.conf[i])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 1)
        cv2.putText(frame, f"fish {conf:.2f}", (x1, y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

    count = len(results[0].boxes)
    cv2.putText(frame, f"Total: {count}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    cv2.imshow("Koi Detection", frame)

    # เช็คว่ามีการกดปุ่ม Esc (27) หรือไม่
    key = cv2.waitKey(1) & 0xFF
    if key == 27: 
        print("Closing program...")
        break

cap.release()
cv2.destroyAllWindows()

ขนาด: 0x0, FPS: 0.0


In [2]:
import cv2
from ultralytics import YOLO

model = YOLO(r"runs/detect/koi-detect/exp_real_100/weights/best.pt")
video_path = r"../data/real_test_data/video.mp4"

cap = cv2.VideoCapture(video_path)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
print(f'ขนาด: {w}x{h}, FPS: {fps}')

PROCESS_EVERY_N = 4  # ประมวลผลทุก 3 เฟรม (ปรับได้)
frame_count = 0
last_results = None  # เก็บผลลัพธ์ล่าสุดไว้แสดงในเฟรมที่ skip

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # ประมวลผลเฉพาะเฟรมที่หารลงตัว
    if frame_count % PROCESS_EVERY_N == 0:
        last_results = model.predict(
            frame,
            conf=0.3,
            verbose=False,
            device=0,
            half=True,        # FP16 ช่วย speed ขึ้นอีก
            imgsz=max(w, h)
        )

    # วาด bounding box จาก last_results (ถ้ามี)
    if last_results is not None:
        for i, box in enumerate(last_results[0].boxes.xyxy):
            x1, y1, x2, y2 = map(int, box)
            conf = float(last_results[0].boxes.conf[i])
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 1)
            cv2.putText(frame, f"fish {conf:.2f}", (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

        count = len(last_results[0].boxes)
        cv2.putText(frame, f"Total: {count}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    cv2.imshow("Koi Detection", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == 27:
        print("Closing program...")
        break

cap.release()
cv2.destroyAllWindows()

ขนาด: 0x0, FPS: 0.0


In [5]:
import cv2
import numpy as np
import os
from ultralytics import YOLO

# --- 1. กำหนดรายการ Path ของวิดีโอที่ต้องการตรวจ (ใส่กี่ไฟล์ก็ได้) ---
VIDEO_PATHS = [
    r"../data/video_real_data/test.mp4",
    r"../data/video_real_data/test2.mp4",
    r"../data/video_real_data/VID_20260302_114947.mp4",
    r"../data/video_real_data/VID_20260302_132631.mp4",
    r"../data/video_real_data/VID_20260302_132743.mp4",
]

# Change Model HERE
# MODEL_NAME = "exp_v1_remake_e125_b12"
# MODEL_NAME = "exp_v1"
# MODEL_NAME = "exp_v2" #v2 No2
# MODEL_NAME = "exp_v2_100" #v2 No1
# MODEL_NAME = "exp_v2_200" #v2 No3
MODEL_NAME = "exp_v2_remake_e100_b8" #v2 remake No1 !!!
# MODEL_NAME = "exp_v2_remake_e150_b8" #v2 remake No2 
# MODEL_NAME = "exp_v3_e50_b8" #v3 Not Work!
# MODEL_NAME = "exp_v4_e50_b8" #v4
# MODEL_NAME = "exp_real_e50_b8" #Real Data No!
# MODEL_NAME = "exp_real_e20_b8" #Real Data No!
# MODEL_NAME = "exp_realbup_e20_b8" #Real Data No!
# MODEL_NAME = "exp_real_100" #Real Data


MODEL_PATH = f"runs/detect/koi-detect/{MODEL_NAME}/weights/best.pt" # v1 Best?
GAMMA_VALUE = 0.3    # ปรับลดแสง (ถ้าน้ำใสปกติใช้ 1.0, ถ้าแสงจ้าลดเหลือ 0.6-0.8)
PROCESS_EVERY_N = 4  # ข้ามเฟรมเพื่อความเร็ว

def adjust_gamma(image, gamma=1.0):
    invGamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** invGamma) * 255
                      for i in np.arange(0, 256)]).astype("uint8")
    return cv2.LUT(image, table)

# โหลด Model ครั้งเดียวใช้ยาวๆ
model = YOLO(MODEL_PATH)

for path in VIDEO_PATHS:
    # ตรวจสอบก่อนว่าไฟล์มีอยู่จริงไหม
    if not os.path.exists(path):
        print(f"❌ ไม่พบไฟล์: {path}")
        continue

    print(f"🚀 กำลังเริ่มตรวจ: {os.path.basename(path)}")
    cap = cv2.VideoCapture(path)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    frame_count = 0
    last_results = None

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        frame_count += 1
        
        # 1. ลดแสง/ปรับ Contrast (Preprocessing)
        processed_frame = adjust_gamma(frame, gamma=GAMMA_VALUE)

        # 2. ส่งให้ AI ตรวจสอบ (Skip frames ตามที่ตั้งไว้)
        if frame_count % PROCESS_EVERY_N == 0:
            last_results = model.predict(
                processed_frame, 
                conf=0.3, 
                verbose=False, 
                device=0, 
                half=True, 
                imgsz=max(w, h)
            )

        # 3. วาดผลลัพธ์
        if last_results is not None:
            for i, box in enumerate(last_results[0].boxes.xyxy):
                x1, y1, x2, y2 = map(int, box)
                conf = float(last_results[0].boxes.conf[i])
                cv2.rectangle(processed_frame, (x1, y1), (x2, y2), (0, 255, 0), 1)
                cv2.putText(processed_frame, f"koi {conf:.2f}", (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
            
            # แสดงจำนวนปลาที่นับได้มุมซ้ายบน
            cv2.putText(processed_frame, f"Detected: {len(last_results[0].boxes)} : {MODEL_NAME}", 
                        (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        # แสดงผลหน้าจอ
        cv2.imshow("Manual Path Detection", processed_frame)

        # กด ESC เพื่อข้ามไปคลิปถัดไป, กด 'q' เพื่อเลิกทำงาน
        key = cv2.waitKey(1) & 0xFF
        if key == 27: break
        if key == ord('q'):
            cap.release()
            cv2.destroyAllWindows()
            exit()

    cap.release()

cv2.destroyAllWindows()
print("✅ ประมวลผลครบทุกไฟล์ในรายการแล้ว")

🚀 กำลังเริ่มตรวจ: test.mp4
🚀 กำลังเริ่มตรวจ: test2.mp4
🚀 กำลังเริ่มตรวจ: VID_20260302_114947.mp4
🚀 กำลังเริ่มตรวจ: VID_20260302_132631.mp4
🚀 กำลังเริ่มตรวจ: VID_20260302_132743.mp4
✅ ประมวลผลครบทุกไฟล์ในรายการแล้ว
